<a href="https://colab.research.google.com/github/Musamehar/ML_Intership/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### 1. Plain Words Framing
For SEO managers allocating weekly refresh resources, we rank mature content items using a transparent multi-signal baseline score:
* **Visibility Weight (50%):** Higher 90-day search impressions mean higher traffic impact at risk.
* **Staleness Risk (35%):** Content older than 180 days faces higher decay probability.
* **Position Opportunity (15%):** Pages in positions 1–10 receive higher priority due to page-one ranking real estate.

### 2. Reason Codes & Action Labels
* `stale_visible_decay`: Content age $\ge 180$ days with 90-day impressions $\ge 500$ showing active/potential decay.
* `page_one_decay_risk`: Content in top 10 search positions with content age $\ge 180$ days.
* `low_ctr_opportunity`: High impression visibility with below-average CTR ($< 0.5\%$).

**Action Label:** `content_refresh_audit`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Setup repository pathing safely for Colab & local envs
if not os.path.exists('ML_Intership') and not Path('data/raw/content_refresh_anonymized.csv').exists():
    !git clone https://github.com/Musamehar/ML_Intership.git

possible_paths = [
    Path('ML_Intership/data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv')
]

data_path = next((p for p in possible_paths if p.exists()), None)
df = pd.read_csv(data_path)

# 2. Filter qualified mature content per FlyRank pipeline rules
df_clean = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id').copy()

# 3. Calculate baseline score components (0.0 - 100.0 scale)
log_imp = np.log1p(df_clean['impressions_90d'])
visibility_score = (log_imp - log_imp.min()) / (log_imp.max() - log_imp.min())
age_risk = np.where(df_clean['content_age_days'] >= 180, 1.0, df_clean['content_age_days'] / 180.0)
pos_opportunity = np.where((df_clean['avg_position'] > 0) & (df_clean['avg_position'] <= 10), 1.0, 0.5)

df_clean['baseline_score'] = (0.50 * visibility_score + 0.35 * age_risk + 0.15 * pos_opportunity) * 100.0
df_clean['baseline_score'] = df_clean['baseline_score'].round(2)

# 4. Assign Reason Codes
def assign_reason_code(row):
    if row['content_age_days'] >= 180 and row['impressions_90d'] >= 500:
        return 'stale_visible_decay'
    elif 0 < row['avg_position'] <= 10 and row['content_age_days'] >= 180:
        return 'page_one_decay_risk'
    else:
        return 'general_refresh_candidate'

df_clean['reason_code'] = df_clean.apply(assign_reason_code, axis=1)
df_clean['action_label'] = 'content_refresh_audit'

# 5. Sort & Export Ranked Queue CSV
ranked_queue = df_clean.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

output_cols = ['content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label',
               'impressions_90d', 'clicks_90d', 'avg_position', 'content_age_days']
export_df = ranked_queue[output_cols]

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('../work/outputs', exist_ok=True)

export_path = 'work/outputs/baseline_action_score.csv'
export_df.to_csv(export_path, index=False)
print(f"SUCCESS: Written {len(export_df):,} ranked rows to '{export_path}'.")
export_df.head(5)

Cloning into 'ML_Intership'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 119 (delta 34), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.85 MiB | 8.80 MiB/s, done.
Resolving deltas: 100% (34/34), done.
SUCCESS: Written 30,000 ranked rows to 'work/outputs/baseline_action_score.csv'.


,content_id,client_id,baseline_score,reason_code,action_label,impressions_90d,clicks_90d,avg_position,content_age_days
0,content_aaef01a50def,client_19581e27de,100.00,stale_visible_decay,content_refresh_audit,517109,1270,5.4,445
1,content_5fe46e04994d,client_4e07408562,100.00,stale_visible_decay,content_refresh_audit,517715,741,4.2,537
2,content_8c19996aa890,client_4e07408562,99.93,stale_visible_decay,content_refresh_audit,509252,785,2.5,445
3,content_4c36c775b818,client_4e07408562,99.55,stale_visible_decay,content_refresh_audit,463103,1889,2.3,445
4,content_1a9e894be2e2,client_19581e27de,99.12,stale_visible_decay,content_refresh_audit,416180,944,4.0,482


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# Generate Top-20 programmatic review table with "what would make it wrong" explanations
top20 = export_df.head(20).copy()

print("=" * 80)
print("TOP-20 SKEPTIC'S EYE REVIEW")
print("=" * 80)

for idx, row in top20.iterrows():
    print(f"Rank {idx+1:02d} | ID: {row['content_id']} | Client: {row['client_id']} | Score: {row['baseline_score']}")
    print(f"   Action: {row['action_label']} | Reason: {row['reason_code']}")
    print(f"   Stats: Imp={row['impressions_90d']:,}, Clicks={row['clicks_90d']:,}, Pos={row['avg_position']}, Age={row['content_age_days']}d")
    print(f"   Confidence Note: High visibility ({row['impressions_90d']:,} imp) justifies audit priority.")
    print(f"   What would make it wrong: If traffic dip is driven by SERP layout shifts (AI Overviews) or seasonal demand rather than content decay.\n")

TOP-20 SKEPTIC'S EYE REVIEW
Rank 01 | ID: content_aaef01a50def | Client: client_19581e27de | Score: 100.0
   Action: content_refresh_audit | Reason: stale_visible_decay
   Stats: Imp=517,109, Clicks=1,270, Pos=5.4, Age=445d
   Confidence Note: High visibility (517,109 imp) justifies audit priority.
   What would make it wrong: If traffic dip is driven by SERP layout shifts (AI Overviews) or seasonal demand rather than content decay.

Rank 02 | ID: content_5fe46e04994d | Client: client_4e07408562 | Score: 100.0
   Action: content_refresh_audit | Reason: stale_visible_decay
   Stats: Imp=517,715, Clicks=741, Pos=4.2, Age=537d
   Confidence Note: High visibility (517,715 imp) justifies audit priority.
   What would make it wrong: If traffic dip is driven by SERP layout shifts (AI Overviews) or seasonal demand rather than content decay.

Rank 03 | ID: content_8c19996aa890 | Client: client_4e07408562 | Score: 99.93
   Action: content_refresh_audit | Reason: stale_visible_decay
   Stats: Imp

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks & Failure Modes
1. **Evergreen Core Pages:** High-impression hub pages that are naturally old (`content_age_days` > 500) but remain factually accurate and search-dominant.
2. **Seasonal Demand Surges:** Pages experiencing calendar off-season traffic drops incorrectly prioritized purely on age and past impression volume.
3. **SERP Disruption:** Pages maintaining top 3 rank positions where click loss is caused by Google introducing AI answers or paid ads above organic results.

### Leakage Verification Check
* **No Future Metrics:** Features depend exclusively on trailing 90-day historical window data.
* **No Target Leakage:** `trend_pct` and `trend_direction` are strictly omitted from scoring calculations.
* **No Synthetic Product Flags:** No application-calculated scores (`health_score`, `priority_score`) were fed into feature engineering.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.